In [1]:
!pip install konlpy
!pip install pandasql
!pip install gensim

In [2]:
import pandas as pd
from konlpy.tag import Okt
import pandasql as psql
from gensim.models import Word2Vec
from gensim.corpora.dictionary import Dictionary
import gensim.models.ldamodel
from gensim.models.coherencemodel import CoherenceModel
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation

from sklearn.metrics.pairwise import cosine_similarity # 코사인 유사도 측정
from sklearn.preprocessing import MinMaxScaler # 정규화

In [3]:
blog_data = pd.read_csv('total_blog(12_3_49).csv')
blog_data.head()

,title,text,view_7days,date
0,"대한민국 교육의 글로벌 전략, 지금 어디까지 왔을까?","최근 국제학생 유치부터 아시아 연합 교육,\r\n\r\n그리고 콘텐츠 산업 특화 전...",3,2025. 8. 4
1,"작은 학점으로 커리어를 설계하다, 마이크로디그리와 디지털 배지의 만남",실무 역량을 키우는 '마이크로디그리' 시대의 시작\r\n\r\n전공만으로는 부족함을...,4,2025. 8. 5
2,"지역 인재 양성을 위한 새로운 해답, '디지털 배지'","""대학의 역할이 지식 전달을 넘어, 지역과 산업을 연결하는 실행 거점으로 진화하고 ...",8,2025. 8. 6
3,2025 자격증취득장려금 신청방법 준비서류 혜택 총정리,오늘은 자격증 취득을 준비하시는 \n\n청년분들께 꼭 필요한 제도를 \n\n소개해드...,19,2025. 8. 7
4,국내 대학의 마이크로디그리 운영 방식은 어떨까요?,"변하는 대학 교육, 마이크로디그리가 해답입니다.\r\n\r\n마이크로디그리(Micr...",12,2025. 8. 8


### 함수 정의

In [4]:
# 불용어 리스트
stopwords = ['디지털', '디지털배지', '오픈배지', '배지']
extra_stopwords = ['이자', '이란', '널리', '더욱', '동해', '커스터',
                   '거나', '마치', '마이', '라면', '예전', '높이',
                   '대신', '만약', '점점', '최근', '정작', '먼저',
                   '진짜', '가시', '최신', '부문', '단지', '본격',
                   '대한', '서나', '여러', '블록', '금은', '언제',
                   '최소', '램폴린', '저희', '그동안', '아무',
                   '차근차근', '금도', '지난', '카카오', '이그',
                   '다음', '외식', '주무', '우리', '무부', '달라',
                   '처란', '스스로', '일반', '주체', '반대']


# 명사 추출 함수
okt = Okt()

def extract_nouns(text):
    tokens = okt.pos(str(text))
    nouns = []
    for w, t in tokens:
        if t == 'Noun' and len(w) >= 2:
            if w not in stopwords and w not in extra_stopwords:
                nouns.append(w)
    return nouns


# Word2Vec 연관어 필터링 함수
def is_valid_noun(word):
    # 1. 길이 및 유효명사 체크
    if not isinstance(word, str) or len(word) < 2 or word.isdigit():
        return False

    # 2. 불용어 제거
    if word in stopwords or word in extra_stopwords:
        return False

    # 3. 명사 여부 확인
    pos = okt.pos(word)
    if len(pos) == 0 or pos[0][1] != 'Noun':
        return False

    return True

### 데이터 전처리

In [5]:
# 날짜 전처리
blog_data['date'] = blog_data['date'].str.replace('.', '-', regex=False).str.replace(' ', '', regex=False)
blog_data['date'] = pd.to_datetime(blog_data['date'], format='%Y-%m-%d')


# 데이터 정제
query = 'SELECT title, text, view_7days, date FROM blog_data WHERE view_7days is not null'
first_week_data = psql.sqldf(query, locals())


# first_week_data 전처리
first_week_data['view_7days'] = first_week_data['view_7days'].astype(int)


# 명사 추출
first_week_data['nouns'] = first_week_data['text'].apply(extract_nouns)

### 시간 및 고성과 그룹 필터링 (상위 25%)

In [6]:
# 분석 시작 날짜
start_date = '2025-08-04'
recent_data = first_week_data[first_week_data['date'] >= start_date].copy()

# 조회수 상위 25% 기준값 계산
quantile_25 = recent_data['view_7days'].quantile(0.25)

# 상위 25% 데이터 필터링
high_performance_data = recent_data[recent_data['view_7days'] >= quantile_25]

print("✅ 데이터 필터링 결과")
print(f"필터링 기준: {start_date} 이후, 조회수 {quantile_25:.0f}회 이상 (상위 25%)")
print(f"분석 게시물 수 : {len(high_performance_data)}개")

✅ 데이터 필터링 결과
필터링 기준: 2025-08-04 이후, 조회수 4회 이상 (상위 25%)
분석 게시물 수 : 37개


### Word2Vec 모델 학습

In [7]:
w2v_sentences = high_performance_data['nouns'].tolist()

# min_count = 1로 설정하여 한 번 등장한 단어(신규 키워드 후보)도 포함
model = Word2Vec(
    sentences=w2v_sentences,
    vector_size=100,
    window=5,
    min_count=1,
    sg=1,
    epochs=30
)

print(f"Word2Vec 모델 재학습 완료 (학습 단어 수: {len(model.wv.index_to_key)}개)")

Word2Vec 모델 재학습 완료 (학습 단어 수: 2051개)


### **[추가]** 잡음 제거

> no_below = 2, no_above = 0.9 <br>
→ 이렇게 했을 때 몇 개의 단어가 남게 되는지 확인 필수
<br>
<br>
확인 후 파라미터 추가 조정




In [8]:
tokenized_docs = high_performance_data['nouns'].tolist()
dictionary = Dictionary(tokenized_docs)

dictionary.filter_extremes(no_below=2, no_above=0.90)
print(f"잡음 제거 후 dictionary 크기 : {len(dictionary)}개")

잡음 제거 후 dictionary 크기 : 1032개


### 최적 k값 탐색 (Coherence Score 도입)

In [9]:
# doc2bow : 단어와 각 단어의 빈도수를 튜플 형태로 만들어서 저장
corpus_gensim = [dictionary.doc2bow(text) for text in tokenized_docs]

coherence_scores = []
k_values = range(5, 11)

print("최적 k값 탐색 (Coherence Score)")

for k in k_values:
    # Gensim LDA 모델 학습
    lda_model_gensim = gensim.models.ldamodel.LdaModel(
        corpus=corpus_gensim, # corpus = 변환된 특성 벡터, dictionary.doc2bow의 실행 결과
        id2word=dictionary, # id2word = 참조할 특성 집합, dictionary 객체
        num_topics=k, # num_topics = 토픽의 수
        random_state=42,
        passes=10 # passes = 전체 corpus를 몇 번 반복해서 학습할지
    )

    # Coherence Score 계산
    coherence_model_lda = CoherenceModel(
        model=lda_model_gensim, # model = 사전 학습된 모델 객체 전달
        texts=tokenized_docs, # texts = 토큰화된 결과
        dictionary=dictionary, # dictionary = Gensim의 dictionary
        coherence='c_v' # coherence - u_mass(반드시 인수로 corpus가 있어야 함)
                        #           - c_v(반드시 인수로 texts가 있어야 함)
    )
    coherence_scores.append(coherence_model_lda.get_coherence())
    print(f"K={k}, Coherence Score: {coherence_scores[-1]:.4f}")

최적 k값 탐색 (Coherence Score)
K=5, Coherence Score: 0.4497
K=6, Coherence Score: 0.4196
K=7, Coherence Score: 0.4463
K=8, Coherence Score: 0.4679
K=9, Coherence Score: 0.3831
K=10, Coherence Score: 0.4139


### 최적 k값 정교화

In [10]:
gains = np.diff(coherence_scores) # Coherence_scores의 변화량
k_gains = k_values[1:] # k=6부터의 변화량

# 문서 수(120±)에 적합한 최소 변화량 임계값 설정
MIN_MARGINAL_GAIN = 0.005

# 플래그 및 초기값 설정
elbow_found = False
optimal_k_refined = k_values[np.argmax(coherence_scores)] # 기본값은 최대 점수를 준 K
print('\n[STEP] Coherence 변화율 분석')

for i, gain in enumerate(gains):
    k_current = k_gains[i]
    print(f"K={k_current}: 이전 K 대비 증가량: {gain:.4f}")

    # 증가량이 임계값 미만으로 떨어진 지점 (Elbow Point)
    if gain < MIN_MARGINAL_GAIN:
        optimal_k_refined = k_values[i]
        elbow_found = True
        print(f" -> 증가량이 {MIN_MARGINAL_GAIN:.4f} 미만으로 떨어져, K={optimal_k_refined}를 Elbow Point로 결정")

# 최종 k값 결정
if not elbow_found:
    print(f" -> 모든 K에서 충분한 증가량이 관찰되어, 최대 Coherence Score ({coherence_scores[np.argmax(coherence_scores)]:.4f})를 달성한 K={optimal_k_refined}를 최종 선택")
    optimal_k = optimal_k_refined # max_score_k와 동일
else:
    optimal_k = optimal_k_refined

print(f"\n✅ 정교화된 최적의 토픽 개수(K): {optimal_k}")


[STEP] Coherence 변화율 분석
K=6: 이전 K 대비 증가량: -0.0301
 -> 증가량이 0.0050 미만으로 떨어져, K=5를 Elbow Point로 결정
K=7: 이전 K 대비 증가량: 0.0267
K=8: 이전 K 대비 증가량: 0.0216
K=9: 이전 K 대비 증가량: -0.0848
 -> 증가량이 0.0050 미만으로 떨어져, K=8를 Elbow Point로 결정
K=10: 이전 K 대비 증가량: 0.0308

✅ 정교화된 최적의 토픽 개수(K): 8


### 최종 LDA 모델 학습

In [11]:
corpus_sklearn = high_performance_data['nouns'].apply(lambda x: ' '.join(x)).tolist()
vectorizer_lda = TfidfVectorizer(stop_words=stopwords + extra_stopwords)
tfidf_matrix = vectorizer_lda.fit_transform(corpus_sklearn)
feature_names = vectorizer_lda.get_feature_names_out()


# 최종 LDA 모델 학습 (최적 k 적용)
lda_model_final = LatentDirichletAllocation(
    n_components=optimal_k,
    random_state=42,
    max_iter=15
)
lda_model_final.fit(tfidf_matrix)


# 토픽 출력 함수
def display_topics(model, feature_names, no_top_words):
    topics = []
    for topic in model.components_:
        top_words = [feature_names[i]
                     for i in topic.argsort()[:-no_top_words -1 : -1]]
        topics.append(top_words)
    return topics

top_keywords_by_topic = display_topics(lda_model_final, feature_names, 10)

### [추가] 주제 겹침 방지


유사도 0.7 이상인 토픽은 겹치는 주제로 간주하여
새로운 아이디어 구상에서 제외



In [12]:
# 유사 토픽 필터링 함수 정의

def filter_similar_topics(model, similarity_threshold=0.7):
    # components_를 행별 합이 1이 되도록 정규화
    # 이를 통해 확률 분포 기반 유사도를 정확하게 계산
    topic_vectors = model.components_ / model.components_.sum(axis=1, keepdims=True)

    # 1. 코사인 유사도 행렬 계산
    similarity_matrix = cosine_similarity(topic_vectors)

    # 2. 제거할 토픽 인덱스 찾기
    topics_to_remove = set()
    num_topics = similarity_matrix.shape[0]

    # 상삼각 행렬만 확인(i < j)
    for i in range(num_topics):
        for j in range(i+1, num_topics):
            # 유사도 임계값 초과 & 제거 대상으로 등록되지 않은 토픽
            if similarity_matrix[i, j] >= similarity_threshold and i not in topics_to_remove and j not in topics_to_remove:
                # 둘 중 토픽 순위가 낮은(Coherence가 낮거나, 임의로 후순위) 토픽을 제거 대상으로 선택
                # 여기서는 임의로 인덱스가 높은(나중에 추출된) j를 제거
                topics_to_remove.add(j)
                print(f"경고 : Topic {i+1}와 Topic {j+1}의 유사도가 {similarity_matrix[i, j]:.2f}로 높아 (임계값, 0.7), Topic {j+1}를 분석에서 제외함")

    # 4. 필터링된 토픽 인덱스 리스트 생성
    filtered_indices = [i for i in range(num_topics) if i not in topics_to_remove]

    return filtered_indices, topics_to_remove

filtered_topic_indices, removed_topics = filter_similar_topics(lda_model_final)

경고 : Topic 1와 Topic 2의 유사도가 0.73로 높아 (임계값, 0.7), Topic 2를 분석에서 제외함
경고 : Topic 1와 Topic 3의 유사도가 0.99로 높아 (임계값, 0.7), Topic 3를 분석에서 제외함
경고 : Topic 1와 Topic 4의 유사도가 0.99로 높아 (임계값, 0.7), Topic 4를 분석에서 제외함
경고 : Topic 1와 Topic 5의 유사도가 0.97로 높아 (임계값, 0.7), Topic 5를 분석에서 제외함
경고 : Topic 1와 Topic 6의 유사도가 0.98로 높아 (임계값, 0.7), Topic 6를 분석에서 제외함
경고 : Topic 1와 Topic 7의 유사도가 0.99로 높아 (임계값, 0.7), Topic 7를 분석에서 제외함
경고 : Topic 1와 Topic 8의 유사도가 0.99로 높아 (임계값, 0.7), Topic 8를 분석에서 제외함


### [추가] 동적 추천 개수 조정

In [13]:
# 1. 토픽별 문서 수 계산

doc_topic_distribution = lda_model_final.transform(tfidf_matrix)
dominant_topic_indices = doc_topic_distribution.argmax(axis=1)
doc_counts_per_topic = np.bincount(dominant_topic_indices, minlength=optimal_k)

In [14]:
# 2.토픽별 응집도 (Coherence Score / Quality) 계산
# Sklearn LDA 결과(Top N Words)를 Gensim CoherenceModel에 사용

TOP_N_WORDS_FOR_TOPIC_LABEL = 10

topic_word_lists = []
for idx, topic in enumerate(lda_model_final.components_):
    # LDA top N 단어 추출
    words = [feature_names[i] for i in topic.argsort()[:-TOP_N_WORDS_FOR_TOPIC_LABEL-1 : -1]]
    # dictionary 안에 있는 단어만 필터링
    valid_words = [w for w in words if w in dictionary.token2id]

    if len(valid_words) == 0:
        print(f"주의: Topic {idx+1}의 단어가 dictionary에 없어 Coherence 계산에서 제외됨")
    else:
        topic_word_lists.append(valid_words)

# 빈 토픽이 없으면 Coherence 계산 진행
if len(topic_word_lists) == 0:
    raise ValueError("모든 토픽 단어가 dictionary에 없어서 Coherence 계산 불가")
else:
    coherence_model = CoherenceModel(
        topics=topic_word_lists,
        texts=tokenized_docs,
        dictionary=dictionary,
        coherence='c_v'
    )

    # 각 토픽별 응집도 점수를 가져옴
    coherence_scores_per_topic = coherence_model.get_coherence()
    print(f"Coherence 계산 완료 (유효 토픽 수 : {len(topic_word_lists)})")
    print(f"전체 Coherence Score = {coherence_scores_per_topic:.4f}")

주의: Topic 1의 단어가 dictionary에 없어 Coherence 계산에서 제외됨
주의: Topic 3의 단어가 dictionary에 없어 Coherence 계산에서 제외됨
주의: Topic 4의 단어가 dictionary에 없어 Coherence 계산에서 제외됨
주의: Topic 7의 단어가 dictionary에 없어 Coherence 계산에서 제외됨
주의: Topic 8의 단어가 dictionary에 없어 Coherence 계산에서 제외됨
Coherence 계산 완료 (유효 토픽 수 : 3)
전체 Coherence Score = 0.7218


In [15]:
# 3. 필터링된 토픽에 대해 최종 중요도 점수 계산

overall_coherence  = coherence_model.get_coherence()

topic_data_list = []
for original_idx in filtered_topic_indices:
    topic_data_list.append({
        'original_idx' : original_idx,
        'doc_count' : doc_counts_per_topic[original_idx],
        'coherence_score' : overall_coherence
    })

topic_df = pd.DataFrame(topic_data_list)
if len(topic_df) == 0:
    print('분석할 토픽이 없습니다.')
    exit()

# 정규화
scaler = MinMaxScaler()

topic_df['norm_doc_count'] = scaler.fit_transform(topic_df[['doc_count']])[:, 0]
# coherence score가 높을수록 좋으므로 그래도 사용
topic_df['norm_coherence'] = scaler.fit_transform(topic_df[['coherence_score']])[:, 0]

# 최종 중요도 점수 계산 : Volume(문서 수) * Quality(Coherence)
topic_df['importance_score'] = topic_df['norm_doc_count'] * topic_df['norm_coherence']

In [16]:
# 4. 동적 추천 개수 기준 정의 및 4단계 계층화
importance_scores = topic_df['importance_score'].tolist()

# 4단계 분류를 위한 분위수 (25%, 50%, 75%)
q25_imp = np.quantile(importance_scores, 0.25)
q50_imp = np.quantile(importance_scores, 0.50)
q75_imp = np.quantile(importance_scores, 0.75)

# 중요도 점수에 따라 추천 키워드 개수를 결정하는 함수 (4단계)
def get_dynamic_recommend_count_refined(score):
    if score > q75_imp:
        return 10 # 상위 25% (매우 중요)
    elif score > q50_imp:
        return 8 # 상위 25~50% (중요)
    elif score > q25_imp:
        return 5 # 하위 50~75% (일반)
    else:
        return 3 # 하위 25% (낮은 중요도)

recommended_keywords_by_topic = {}

for idx, row in topic_df.iterrows():
    topic_idx = int(row['original_idx'])
    importance_score = row['importance_score']

    # 추천키워드 개수 결정
    num_keywords = get_dynamic_recommend_count_refined(importance_score)

    # top_keywords_by_topic에서 상위 num_keywords 추출
    recommended_keywords = top_keywords_by_topic[topic_idx][:num_keywords]

    recommended_keywords_by_topic[topic_idx] = recommended_keywords

for topic_idx, keywords in recommended_keywords_by_topic.items():
    print(f"Topic {topic_idx+1} 추천 키워드: {keywords}")

Topic 1 추천 키워드: ['의무교육', '노인', '요양']


### Word2Vec 연계 분석 (새로운 주제 아이디어 발굴)

In [17]:
print("✅ Word2Vec 기반 토픽별 새로운 연관 키워드 추천")

recommendations_by_topic = []

for topic_idx, top_words in zip(filtered_topic_indices, top_keywords_by_topic):
    # 해당 토픽 중요도
    topic_importance_score = topic_df.loc[topic_df['original_idx'] == topic_idx, 'importance_score'].values[0]

    # 핵심 키워드 개수 동적 결정
    num_top_words = get_dynamic_recommend_count_refined(topic_importance_score)
    search_words = top_words[:num_top_words]

    related_words = []

    for kw in search_words:
        try:
            # Word2Vec 연관어 탐색 (10개 후보)
            candidates = model.wv.most_similar(kw, topn=10)

            # 유효성 필터링
            valid_words = [w for w, sim in candidates if is_valid_noun(w) and w not in search_words]

            # 연관키워드 개수도 핵심 키워드 개수에 비례하여 동적 결정(최대 2개)
            num_related_per_kw = max(1, min(2, num_top_words//5)) # 핵심 키워드 5개 이상이면 2개, 아니면 1개
            related_words.extend(valid_words[:num_related_per_kw])

        except KeyError:
            continue

    # 중복 제거 및 최종 추천 목록 (최대 8개)
    unique_related_words = list(set(related_words))[:8]

    recommendations_by_topic.append({
        'Topic ID' : topic_idx + 1,
        '핵심키워드' : ', '.join(search_words),
        '추천연관키워드' : ', '.join(unique_related_words)
    })

recommend_df_topic = pd.DataFrame(recommendations_by_topic)
print(recommend_df_topic.to_string())

✅ Word2Vec 기반 토픽별 새로운 연관 키워드 추천
   Topic ID         핵심키워드 추천연관키워드
0         1  의무교육, 노인, 요양  목욕, 인권


### [추가] 트렌드 반영

In [18]:
import json
import numpy as np
import time
import requests
from datetime import datetime, timedelta

In [27]:
def min_max_normalize(scores):
    scores = np.array(scores, dtype=float)
    min_val = np.min(scores)
    max_val = np.max(scores)

    # 모든 점수가 같은 경우 (분모가 0이 되는 것을 방지)
    if max_val == min_val:
        return np.full_like(scores, 0.5)

    normalized_scores = (scores - min_val) / (max_val - min_val)
    return normalized_scores

In [29]:
def sum_normalize(scores):
    scores = np.array(scores, dtype=float)
    total_sum = np.sum(scores)

    if total_sum == 0:
        return np.zeros_like(scores)

    normalized_scores = scores / total_sum
    return normalized_scores

In [28]:
# 1. 네이버 데이터랩 API 설정
# https://developers.naver.com/products/service-api/datalab/datalab.md
# 위 주소에서 오픈 API 신청 가능

NAVER_CLIENT_ID = 'NAVER_CLIENT_ID'
NAVER_CLIENT_SECRET = 'NAVER_CLIENT_SECRET'
NAVER_API_URL = 'https://openapi.naver.com/v1/datalab/search'

In [21]:
# 2. 토픽별 핵심 키워드

TOPIC_KEYWORDS = {}

for row in recommendations_by_topic:
    topic_id = row['Topic ID'] -1 # Topic ID를 0부터 시작하도록 변환
    keywords = row['핵심키워드'].split(', ')
    TOPIC_KEYWORDS[topic_id] = keywords

NUM_TOPICS = len(TOPIC_KEYWORDS)
print(f'앞서 추출된 토픽 개수 : {NUM_TOPICS}')

앞서 추출된 토픽 개수 : 1


In [22]:
# 3. 네이버 데이터랩 API 실제 호출 함수
# 단일 키워드에 대해 네이버 데이터랩 API를 호출하고 평균 상대 검색량을 반환
# 데이터 왜곡을 막기 위해 항상 단일 키워드로 호출

def fetch_keyword_volume(keyword):
    if not NAVER_CLIENT_ID or not NAVER_CLIENT_SECRET or NAVER_CLIENT_ID == 'NAVER_CLIENT_ID':
        # 인증 정보 없을 시 시뮬레이션 모드(실제값과 다름)
        return np.random.randint(1, 100)
    headers = {
        "X-Naver-Client-Id" : NAVER_CLIENT_ID,
        "X-Naver-Client-Secret" : NAVER_CLIENT_SECRET,
        "Content-Type" : "application/json"
    }

    end_date = datetime.today().strftime('%Y-%m-%d')
    start_date = (datetime.today() - timedelta(days=90)).strftime('%Y-%m-%d')

    # 단일 키워드를 그룹으로 구성 (네이버 API 규정)
    request_body = {
        "startDate" : start_date,
        "endDate" : end_date,
        "timeUnit" : "month",
        "keywordGroups" : [
            {
                "groupName" : keyword,
                "keywords" : [keyword]
            }
        ],
        "device" : "", "gender": "", "ages":[]
    }

    # 호출 제한 고려 : API 호출 전에 1초 지연 시간 추가
    time.sleep(1)

    try:
        response = requests.post(NAVER_API_URL, headers=headers, json=request_body)
        response.raise_for_status()
        result = response.json()

        # 검색량 평균 (토픽의 수요) 계산
        if 'results' in result and result['results'] and 'data' in result['results'][0]:
            data_points = [item['ratio'] for item in result['results'][0]['data']]
            if data_points:
                return np.mean(data_points)
        return 0 # 데이터 추출 실패 시

    except requests.exceptions.RequestException as e:
        print(f"네이버 API 호출 실패 (키워드: {keyword}): {e}")
        # 실패 시 임시로 시뮬레이션 값 반환 (분석 중단 방지)
        return np.random.randint(1, 100)

In [25]:
# 4. 토픽별 평균 검색량 정규화
# 각 토픽의 모든 키워드를 개별 호출 후, 평균 검색량을 기반으로 수요 점수를 계산하고 정규화

def calculate_external_demand_scores(topic_keywords, normalize_method='min_max'):
    raw_scores = []

    # 4-1. 각 토픽별 외부 수요 점수 수집
    for topic_id, keywords in topic_keywords.items():
        total_volume = 0
        keyword_count = 0

        # 키워드별 개별 호출 및 합산 (데이터 왜곡 방지)
        for keyword in keywords:
            volume = fetch_keyword_volume(keyword)
            total_volume += volume
            keyword_count += 1
            print(f"[API] 토픽 {topic_id+1} - '{keyword}': 검색량 {volume:.2f}")

        # 토픽 수요 점수 = 키워드 검색량의 평균
        avg_search_volume = total_volume / keyword_count
        raw_scores.append(avg_search_volume)
        print(f"토픽{topic_id+1} 평균 검색량: {avg_search_volume:.2f}")

    # 4-2. 외부 수요 점수 정규화
    raw_scores_array = np.array(raw_scores)

    if normalize_method == 'min_max':
        normalized_scores = min_max_normalize(raw_scores_array)
        method_name = 'Min-Max 정규화'
    elif normalize_method == 'sum':
        normalized_scores = sum_normalize(raw_scores_array)
        method_name = '합계 기반 정규화'
    else:
        # 정규화를 수행하지 않음(raw score 반환)
        normalized_scores = raw_scores_array
        method_name = '정규화 없음'

    print(f"적용된 정규화 기법 : {method_name}")

    # 4-3. 최종 결과 저장
    external_demand_scores = {i: round(score, 4) for i, score in enumerate(normalized_scores.flatten())}

    return external_demand_scores

In [33]:
# 5. 실행 및 결과 출력

if __name__ == '__main__':
    # Min-Max 정규화 적용
    print("> Min-Max 정규화를 통한 외부 수요 점수 계산")
    final_external_scores_mm = calculate_external_demand_scores(TOPIC_KEYWORDS, normalize_method='min_max')
    print("\n✔ 토픽별 외부 수요 점수 (Min-Max 정규화)")
    sorted_scores_mm = sorted(final_external_scores_mm.items(), key=lambda item: item[1], reverse=True)
    for topic_id, score in sorted_scores_mm:
        print(f"토픽 {topic_id+1} : {score:.4f} (키워드: {', '.join(TOPIC_KEYWORDS[topic_id])})")

    # 합계 기반 정규화 적용
    print("\n\n> 합계 기반 정규화를 통한 외부 수요 점수 계산")
    final_external_scores_sum = calculate_external_demand_scores(TOPIC_KEYWORDS, normalize_method='sum')
    print("\n✔ 토픽별 외부 수요 점수 (합계 기반 정규화)")
    sorted_scores_sum = sorted(final_external_scores_sum.items(), key=lambda item: item[1], reverse=True)
    for topic_id, score in sorted_scores_sum:
        (f"토픽 {topic_id+1} : {score:.4f} (키워드: {', '.join(TOPIC_KEYWORDS[topic_id])})")

    print(f"\n\n참고 : Min-Max 정규화된 점수 합계: {sum(final_external_scores_mm.values()):.4f}")
    print(f"참고 : 합계 기반 정규화된 점수 합계: {sum(final_external_scores_sum.values()):.4f}")

> Min-Max 정규화를 통한 외부 수요 점수 계산
[API] 토픽 1 - '의무교육': 검색량 75.53
[API] 토픽 1 - '노인': 검색량 67.36
[API] 토픽 1 - '요양': 검색량 75.90
토픽1 평균 검색량: 72.93
적용된 정규화 기법 : Min-Max 정규화

✔ 토픽별 외부 수요 점수 (Min-Max 정규화)
토픽 1 : 0.5000 (키워드: 의무교육, 노인, 요양)


> 합계 기반 정규화를 통한 외부 수요 점수 계산
[API] 토픽 1 - '의무교육': 검색량 75.53
[API] 토픽 1 - '노인': 검색량 67.36
[API] 토픽 1 - '요양': 검색량 75.90
토픽1 평균 검색량: 72.93
적용된 정규화 기법 : 합계 기반 정규화

✔ 토픽별 외부 수요 점수 (합계 기반 정규화)


참고 : Min-Max 정규화된 점수 합계: 0.5000
참고 : 합계 기반 정규화된 점수 합계: 1.0000
